In [13]:
import pandas as pd
from pathlib import Path

root = Path("..")
tables = root / "reports" / "tables"

# pega o arquivo mais recente de safra Out/Nov/Dez (independente do ano)
kpi_files = sorted(tables.glob("safra_kpis_*_out_nov_dez.csv"))
if not kpi_files:
    raise FileNotFoundError("Não encontrei safra_kpis_*_out_nov_dez.csv em /reports/tables. Rode: python -m src.03_analysis_safra")

kpi_path = kpi_files[-1]
print("Lendo:", kpi_path.name)

kpi = pd.read_csv(kpi_path)
kpi

Lendo: safra_kpis_2025_out_nov_dez.csv


,mes,n_propostas,valor_emprestado_total,valor_emprestado_medio,taxa_media,score_medio,incident_rate_pct,rebate_share_pct,rebate_spread_medio
0,2025-10,285.0,916942.882,3217.343446,NaN,812.471831,36.842105,0.0,NaN
1,2025-11,207.0,674255.012,3257.270589,NaN,807.922705,20.772947,0.0,NaN
2,2025-12,66.0,255517.725,3871.480682,NaN,802.878788,1.515152,0.0,NaN


In [15]:
meses = list(kpi["mes"].astype(str))
print("Meses no arquivo:", meses)

assert len(meses) == 3, "Era esperado Out/Nov/Dez (3 linhas)."
assert meses[0].endswith("-10") and meses[1].endswith("-11") and meses[2].endswith("-12"), "Meses não são Out/Nov/Dez."

Meses no arquivo: ['2025-10', '2025-11', '2025-12']


In [17]:
kpi["mes"] = kpi["mes"].astype(str)

def pct_change(a, b):
    if a == 0 or pd.isna(a) or pd.isna(b):
        return None
    return (b/a - 1) * 100

cols = ["n_propostas","valor_emprestado_total","taxa_media","score_medio","incident_rate_pct","rebate_share_pct"]
summary = {}

for c in cols:
    oct_v = float(kpi.loc[kpi["mes"].str.endswith("-10"), c].iloc[0])
    nov_v = float(kpi.loc[kpi["mes"].str.endswith("-11"), c].iloc[0])
    dec_v = float(kpi.loc[kpi["mes"].str.endswith("-12"), c].iloc[0])
    summary[c] = {
        "Out": oct_v, "Nov": nov_v, "Dez": dec_v,
        "Out->Nov %": None if pct_change(oct_v, nov_v) is None else round(pct_change(oct_v, nov_v), 2),
        "Nov->Dez %": None if pct_change(nov_v, dec_v) is None else round(pct_change(nov_v, dec_v), 2),
    }

pd.DataFrame(summary).T

,Out,Nov,Dez,Out->Nov %,Nov->Dez %
n_propostas,285.000000,207.000000,66.000000,-27.37,-68.12
valor_emprestado_total,916942.882000,674255.012000,255517.725000,-26.47,-62.10
taxa_media,NaN,NaN,NaN,NaN,NaN
score_medio,812.471831,807.922705,802.878788,-0.56,-0.62
incident_rate_pct,36.842105,20.772947,1.515152,-43.62,-92.71
rebate_share_pct,0.000000,0.000000,0.000000,NaN,NaN


In [19]:
by_files = sorted(tables.glob("safra_*_out_nov_dez_by_*.csv"))
print("Arquivos de quebra encontrados:")
for f in by_files:
    print("-", f.name)

Arquivos de quebra encontrados:
- safra_2025_out_nov_dez_by_Classe_de_Risco_b2e.csv
- safra_2025_out_nov_dez_by_Clinica.csv
- safra_2025_out_nov_dez_by_Estado_Empregatício_Informado.csv
- safra_2025_out_nov_dez_by_Política.csv
- safra_2025_out_nov_dez_by_Política_Rebate.csv
- safra_2025_out_nov_dez_by_score_bucket.csv
- safra_2025_out_nov_dez_by_taxa_bucket.csv


In [21]:
# pega os mais recentes por padrão
pol_files = sorted(tables.glob("safra_*_out_nov_dez_by_Política.csv"))
cli_files = sorted(tables.glob("safra_*_out_nov_dez_by_Clinica.csv"))
score_files = sorted(tables.glob("safra_*_out_nov_dez_by_score_bucket.csv"))
taxa_files = sorted(tables.glob("safra_*_out_nov_dez_by_taxa_bucket.csv"))

pol = pd.read_csv(pol_files[-1]) if pol_files else None
cli = pd.read_csv(cli_files[-1]) if cli_files else None
score = pd.read_csv(score_files[-1]) if score_files else None
taxa = pd.read_csv(taxa_files[-1]) if taxa_files else None

(pol.head(15) if pol is not None else "Sem Política"), (cli.head(15) if cli is not None else "Sem Clinica")

(       mes  dim_value  n_propostas  valor_emprestado_total  \
 0  2025-10        NaN        285.0              916942.882   
 1  2025-11        NaN        207.0              674255.012   
 2  2025-12        NaN         66.0              255517.725   
 
    valor_emprestado_medio  taxa_media  score_medio  incident_rate_pct  \
 0             3217.343446         NaN   812.471831          36.842105   
 1             3257.270589         NaN   807.922705          20.772947   
 2             3871.480682         NaN   802.878788           1.515152   
 
    rebate_share_pct  rebate_spread_medio  
 0               0.0                  NaN  
 1               0.0                  NaN  
 2               0.0                  NaN  ,
         mes                                          dim_value  n_propostas  \
 0   2025-10      PM13275 - {MAIS TODOS} AMOR SAUDE SANTO ANDRE          7.0   
 1   2025-10       PM13366 - {MAIS TODOS} AMOR SAUDE SAO MATEUS          6.0   
 2   2025-10           PM12964 